In [1]:
!pip install -q timm


In [4]:
%%writefile experiment_cross_dataset.py
#!/usr/bin/env python3
"""
============================================================================
CROSS-DATASET GENERALIZATION EXPERIMENT
DualBranchCrackNet — Pavement Crack Detection
============================================================================

Model:      DualBranchCrackNet (ResNet50 + Swin Transformer)
Trained on: 70% of Asphalt 720x720 dataset (stratified split, seed=42)
Tested on:  4 independent datasets (see below)

Label Convention (FIXED — DO NOT CHANGE):
    Cracked     = 0
    Non-cracked = 1

Positive class for binary metrics: Cracked (pos_label=0)

This script performs EVALUATION ONLY.  No retraining is performed.
The existing checkpoint (crackdualnet_best.pth) is used as-is.

No pixel-level IoU, Dice, or SSIM is calculated because these image-
classification datasets do not contain pixel-level ground-truth masks.

The model's heatmap output is labeled as "Predicted Crack Localization
(Model Activation)" — NOT as ground-truth segmentation.

============================================================================
USAGE (Kaggle / Colab / Local):

    python experiment_cross_dataset.py

Or paste sections into notebook cells.  See documentation/README.md for
detailed instructions on uploading datasets and running on Kaggle/Colab.
============================================================================
"""

# ============================================================================
# IMPORTS
# ============================================================================
import os
import sys
import time
import copy
import random
import hashlib
import json
import warnings
import gc
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend (headless servers / Kaggle)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================================
# SECTION 1 — CONFIGURATION & REPRODUCIBILITY
# ============================================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 256                          # Must match training (256x256x3)
BATCH_SIZE = 64 if torch.cuda.is_available() else 8
NUM_WORKERS = 4 if torch.cuda.is_available() else 0
NUM_CLASSES = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

# Label convention — MUST NOT be changed
LABEL_MAP = {0: "Cracked", 1: "Non-cracked"}

# Number of representative visualisation examples per TP/TN/FP/FN category
NUM_VIS_EXAMPLES = 3

# ── Manual path overrides (set these if auto-detection fails) ───────────────
# Leave as None to use auto-detection.
MANUAL_CHECKPOINT   = None   # e.g. "/kaggle/input/my-ckpt/crackdualnet_best.pth"
MANUAL_ASPHALT_720  = None   # root containing Cracked720/ and Uncracked720/
MANUAL_ASPHALT_360  = None   # root containing Cracked360/ and Uncracked360/
MANUAL_SUT_CRACK    = None   # root containing "With Crack/" and "Without Crack/"
MANUAL_SDNET        = None   # root containing Cracked/ and Non-cracked/


# ============================================================================
# SECTION 2 — PATH AUTO-DETECTION
# ============================================================================

def _first_existing(*candidates):
    """Return the first path that actually exists on disk."""
    for p in candidates:
        if p and os.path.exists(p):
            return os.path.abspath(p)
    return None


def _walk_find_dir(roots, must_contain_subdirs):
    """
    Walk *roots* and return the first directory whose immediate children
    include ALL names listed in *must_contain_subdirs* (case-insensitive).
    """
    must_lower = [s.lower() for s in must_contain_subdirs]
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            children_lower = [d.lower() for d in dirnames]
            if all(m in children_lower for m in must_lower):
                return os.path.abspath(dirpath)
    return None


def _walk_find_file(roots, filename):
    """Walk *roots* and return the full path to the first match of *filename*."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            if filename in filenames:
                return os.path.abspath(os.path.join(dirpath, filename))
    return None


def setup_paths():
    """Auto-detect checkpoint and dataset paths for Kaggle / Colab / local."""

    script_dir = os.path.dirname(os.path.abspath(__file__))
    search_roots = []
    # Kaggle
    if os.path.isdir("/kaggle/input"):
        search_roots.append("/kaggle/input")
    # Colab
    if os.path.isdir("/content"):
        search_roots.append("/content")
    if os.path.isdir("/content/drive/MyDrive"):
        search_roots.append("/content/drive/MyDrive")
    # Local (relative to script)
    search_roots.extend([script_dir, os.path.join(script_dir, "..")])

    # ── Checkpoint ──
    ckpt = MANUAL_CHECKPOINT or _walk_find_file(search_roots, "crackdualnet_best.pth")

    # ── Asphalt 720 ──  (must contain Cracked720/ and Uncracked720/)
    a720 = MANUAL_ASPHALT_720 or _walk_find_dir(search_roots, ["Cracked720", "Uncracked720"])

    # ── Asphalt 360 ──  (must contain Cracked360/ and Uncracked360/)
    a360 = MANUAL_ASPHALT_360 or _walk_find_dir(search_roots, ["Cracked360", "Uncracked360"])

    # ── SUT-Crack ──    (must contain "With Crack"/ and "Without Crack"/)
    sut = MANUAL_SUT_CRACK or _walk_find_dir(search_roots, ["With Crack", "Without Crack"])

    # ── SDNET2018 ──    (must contain Cracked/ and Non-cracked/ under Pavements)
    sdnet = MANUAL_SDNET or _walk_find_dir(search_roots, ["Cracked", "Non-cracked"])
    # Disambiguate: if we accidentally found the Asphalt 720 root, skip it
    if sdnet and ("720" in sdnet or "360" in sdnet):
        # Re-search, skipping asphalt dirs
        sdnet = None
        for root in search_roots:
            if not os.path.isdir(root):
                continue
            for dirpath, dirnames, _ in os.walk(root):
                children_lower = [d.lower() for d in dirnames]
                if "cracked" in children_lower and "non-cracked" in children_lower:
                    if "720" not in dirpath and "360" not in dirpath:
                        sdnet = os.path.abspath(dirpath)
                        break
            if sdnet:
                break

    return {
        "checkpoint":  ckpt,
        "asphalt_720": a720,
        "asphalt_360": a360,
        "sut_crack":   sut,
        "sdnet":       sdnet,
    }


# ============================================================================
# SECTION 3 — MODEL ARCHITECTURE  (IDENTICAL TO TRAINING CODE)
# ============================================================================

class DualBranchCrackNet(nn.Module):
    """
    Dual-branch crack detection network (evaluation-only copy).

    Branch 1 : ResNet-50 CNN  (stem → layer1 → layer2 → layer3)
    Branch 2 : Swin-Tiny Transformer  (stage-2 features)
    Fusion   : 1×1 projection → bilinear resize → element-wise addition
    Heads    : classification (2-class softmax) + heatmap (sigmoid, CAM-style)

    Architecture and hyper-parameters are IDENTICAL to the training notebook
    (Cell 1 of train-proposed-only.ipynb).
    """

    def __init__(self, num_classes=2, fused_channels=512,
                 fused_size=16, img_size=256):
        super().__init__()
        self.fused_size = fused_size

        # ── CNN branch: ResNet-50 (weights loaded from checkpoint) ──
        resnet = models.resnet50(weights=None)
        self.cnn_stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3,
        )
        cnn_out_channels = 1024  # layer3 output

        # ── Swin Transformer branch ──
        self.swin = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=False,
            features_only=True,
            out_indices=(2,),
            img_size=img_size,
        )
        swin_out_channels = self.swin.feature_info.channels()[0]

        # ── Projection to common channel dimension ──
        self.cnn_proj  = nn.Conv2d(cnn_out_channels, fused_channels, 1)
        self.swin_proj = nn.Conv2d(swin_out_channels, fused_channels, 1)

        # ── Classification head ──
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(fused_channels, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

        # ── Heatmap head (CAM-style; NOT separately supervised) ──
        self.heatmap_conv = nn.Conv2d(fused_channels, 1, kernel_size=1)

    def forward(self, x):
        # CNN
        f_cnn = self.cnn_proj(self.cnn_stem(x))
        f_cnn = F.interpolate(f_cnn,
                              size=(self.fused_size, self.fused_size),
                              mode="bilinear", align_corners=False)
        # Swin
        f_swin = self.swin(x)[0]
        if f_swin.shape[1] != self.swin_proj.in_channels:
            f_swin = f_swin.permute(0, 3, 1, 2).contiguous()
        f_swin = self.swin_proj(f_swin)
        f_swin = F.interpolate(f_swin,
                               size=(self.fused_size, self.fused_size),
                               mode="bilinear", align_corners=False)
        # Fusion
        fused = f_cnn + f_swin

        logits = self.classifier(fused)
        heatmap = torch.sigmoid(self.heatmap_conv(fused))
        heatmap_full = F.interpolate(heatmap, size=(256, 256),
                                     mode="bilinear", align_corners=False)
        return logits, heatmap_full, fused


def load_model(checkpoint_path):
    """Load DualBranchCrackNet with the trained checkpoint (eval-only)."""
    print(f"\n{'='*70}")
    print("LOADING MODEL")
    print(f"{'='*70}")
    print(f"  Checkpoint : {checkpoint_path}")
    print(f"  File size  : {os.path.getsize(checkpoint_path) / 1e6:.1f} MB")

    model = DualBranchCrackNet(num_classes=NUM_CLASSES, img_size=IMG_SIZE)
    state_dict = torch.load(checkpoint_path, map_location=DEVICE,
                            weights_only=True)
    model.load_state_dict(state_dict, strict=True)
    model = model.to(DEVICE)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Device     : {DEVICE}")
    print(f"  Parameters : {n_params:,}")
    print(f"  Status     : ✓ Loaded successfully (eval mode, strict=True)")
    return model


# ============================================================================
# SECTION 4 — IEAP ENHANCEMENT  &  INFERENCE DATASET
# ============================================================================

def ieap_enhance(bgr_img):
    """
    IEAP enhancement using CLAHE on the L channel of LAB colour space.
    Parameters identical to training:  clipLimit=2.5, tileGridSize=(8,8).
    """
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l_enh = clahe.apply(l)
    enhanced = cv2.merge((l_enh, a, b))
    return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)


class InferenceDataset(Dataset):
    """
    Dataset for inference.  Preprocessing pipeline matches training exactly:
        1. cv2.imread  (BGR)
        2. cv2.resize  to 256×256, INTER_AREA
        3. CLAHE IEAP  enhancement
        4. BGR → RGB
        5. ToTensor + ImageNet normalisation
    No augmentation (evaluation mode).
    """

    def __init__(self, file_paths, file_labels, img_size=256):
        self.file_paths  = file_paths
        self.file_labels = file_labels
        self.img_size    = img_size
        self.transform   = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        label    = self.file_labels[idx]

        bgr = cv2.imread(img_path)
        if bgr is None:
            bgr = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
            print(f"  WARNING: could not read {img_path}")
        bgr = cv2.resize(bgr, (self.img_size, self.img_size),
                         interpolation=cv2.INTER_AREA)
        bgr = ieap_enhance(bgr)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        tensor_img = self.transform(Image.fromarray(rgb))
        return tensor_img, label, img_path


# ============================================================================
# SECTION 5 — DATA LOADING
# ============================================================================

def collect_images(folder_path):
    """Return a sorted list of valid image paths inside *folder_path*."""
    imgs = []
    if not os.path.isdir(folder_path):
        return imgs
    for fname in sorted(os.listdir(folder_path)):
        if fname.lower().endswith(IMG_EXTENSIONS):
            imgs.append(os.path.join(folder_path, fname))
    return imgs


def load_labeled_dataset(root_dir, cracked_folder, uncracked_folder):
    """
    Build (paths, labels) from two class sub-folders.
    Cracked=0,  Non-cracked=1.
    """
    cracked_imgs   = collect_images(os.path.join(root_dir, cracked_folder))
    uncracked_imgs = collect_images(os.path.join(root_dir, uncracked_folder))

    paths  = cracked_imgs + uncracked_imgs
    labels = [0] * len(cracked_imgs) + [1] * len(uncracked_imgs)
    return paths, labels, len(cracked_imgs), len(uncracked_imgs)


def load_asphalt720_split(root_dir):
    """
    Load the Asphalt 720 dataset and reproduce the 70 / 30 stratified split.

    File listing is **sorted** for cross-platform reproducibility.
    train_test_split(test_size=0.30, random_state=42, stratify=labels).

    NOTE:  The original Kaggle training run used os.listdir() whose order is
    filesystem-dependent.  Using sorted() guarantees an identical split on
    every machine, but the exact images may differ from the Kaggle run.
    The checkpoint is frozen, so this is a valid, unbiased evaluation set.
    """
    class_folders = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])

    all_paths, all_labels = [], []
    class_to_idx = {}

    for idx, cname in enumerate(class_folders):
        class_to_idx[cname] = idx
        folder = os.path.join(root_dir, cname)
        imgs = collect_images(folder)           # sorted
        all_paths.extend(imgs)
        all_labels.extend([idx] * len(imgs))

    print(f"  Class mapping : {class_to_idx}")
    print(f"  Total images  : {len(all_paths)}")

    train_paths, test_paths, train_labels, test_labels = train_test_split(
        all_paths, all_labels,
        test_size=0.30, random_state=SEED, stratify=all_labels,
    )
    return train_paths, test_paths, train_labels, test_labels, class_to_idx


# ============================================================================
# SECTION 6 — DATA LEAKAGE VERIFICATION
# ============================================================================

def check_leakage(train_paths, test_datasets):
    """
    Verify zero overlap between the training set and every test set.
    Also flag filename coincidences across datasets (informational only).
    """
    print(f"\n{'='*70}")
    print("DATA LEAKAGE VERIFICATION")
    print(f"{'='*70}")

    train_abs  = set(os.path.abspath(p) for p in train_paths)
    train_base = set(os.path.basename(p) for p in train_paths)
    all_clean  = True

    for ds_name, ds_paths in test_datasets.items():
        test_abs  = set(os.path.abspath(p) for p in ds_paths)
        test_base = set(os.path.basename(p) for p in ds_paths)

        path_overlap = train_abs & test_abs
        name_overlap = train_base & test_base

        if path_overlap:
            print(f"  ✗ LEAKAGE in {ds_name}: "
                  f"{len(path_overlap)} paths overlap with training!")
            all_clean = False
        elif name_overlap:
            print(f"  △ {ds_name}: {len(name_overlap)} shared filenames "
                  f"(different paths — no leakage, name coincidence only)")
        else:
            print(f"  ✓ {ds_name}: 0 shared paths, 0 shared filenames")

    # Cross-test-dataset filename check
    print(f"\n  Cross-dataset filename coincidence check:")
    ds_names = list(test_datasets.keys())
    for i in range(len(ds_names)):
        for j in range(i + 1, len(ds_names)):
            n_i = set(os.path.basename(p) for p in test_datasets[ds_names[i]])
            n_j = set(os.path.basename(p) for p in test_datasets[ds_names[j]])
            shared = n_i & n_j
            tag = f"△ {len(shared)} shared" if shared else "✓ 0 shared"
            print(f"    {tag}  {ds_names[i]} ↔ {ds_names[j]}")

    if all_clean:
        print(f"\n  ✓✓✓ ALL LEAKAGE CHECKS PASSED")
    else:
        print(f"\n  ✗✗✗ DATA LEAKAGE DETECTED — review output above")
        sys.exit(1)

    return all_clean


# ============================================================================
# SECTION 7 — EVALUATION FUNCTION
# ============================================================================

def evaluate_dataset(model, paths, labels, dataset_name, output_dir):
    """
    Run inference on one dataset and produce all metrics / CSVs.

    Returns a dict with metrics and per-image predictions.
    """
    print(f"\n{'='*70}")
    print(f"EVALUATING: {dataset_name}")
    print(f"{'='*70}")

    n_cracked   = sum(1 for l in labels if l == 0)
    n_uncracked = sum(1 for l in labels if l == 1)
    print(f"  Total images   : {len(paths)}")
    print(f"  Cracked   (0)  : {n_cracked}")
    print(f"  Non-crk   (1)  : {n_uncracked}")

    ds     = InferenceDataset(paths, labels, IMG_SIZE)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS,
                        pin_memory=torch.cuda.is_available())

    all_preds, all_labels_out, all_paths_out, all_confs = [], [], [], []

    t0 = time.time()
    with torch.no_grad():
        for bi, (imgs, lbls, img_paths) in enumerate(loader):
            imgs   = imgs.to(DEVICE)
            logits, _, _ = model(imgs)
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            preds  = probs.argmax(axis=1)
            confs  = probs.max(axis=1)

            all_preds.extend(preds.tolist())
            all_labels_out.extend(
                lbls.numpy().tolist() if torch.is_tensor(lbls)
                else list(lbls)
            )
            all_paths_out.extend(img_paths)
            all_confs.extend(confs.tolist())

            if (bi + 1) % 50 == 0 or (bi + 1) == len(loader):
                done = min((bi + 1) * BATCH_SIZE, len(paths))
                print(f"    {done:>6d}/{len(paths)} images processed …")

    elapsed = time.time() - t0
    print(f"  Inference time : {elapsed:.1f}s  "
          f"({len(paths) / max(elapsed, 0.01):.1f} img/s)")

    # ── Metrics (pos_label=0 = Cracked) ──
    acc = accuracy_score(all_labels_out, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels_out, all_preds,
        average="binary", pos_label=0, zero_division=0,
    )
    cm = confusion_matrix(all_labels_out, all_preds, labels=[0, 1])
    report = classification_report(
        all_labels_out, all_preds,
        target_names=["Cracked", "Non-cracked"],
        labels=[0, 1], zero_division=0,
    )

    print(f"\n  {'─'*50}")
    print(f"  {dataset_name} RESULTS")
    print(f"  {'─'*50}")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}  (Cracked, pos_label=0)")
    print(f"  Recall    : {rec:.4f}  (Cracked, pos_label=0)")
    print(f"  F1-score  : {f1:.4f}  (Cracked, pos_label=0)")
    print(f"\n  Confusion Matrix  [rows=true, cols=predicted]")
    print(f"  {'':20s} Pred Cracked   Pred Non-crk")
    print(f"  True Cracked       {cm[0,0]:>10d}   {cm[0,1]:>10d}")
    print(f"  True Non-crk       {cm[1,0]:>10d}   {cm[1,1]:>10d}")
    print(f"\n{report}")

    # ── Per-image CSV ──
    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, f"{dataset_name}_predictions.csv")
    pd.DataFrame({
        "image_path":           all_paths_out,
        "true_label":           all_labels_out,
        "predicted_label":      all_preds,
        "true_label_name":      [LABEL_MAP[l] for l in all_labels_out],
        "predicted_label_name": [LABEL_MAP[p] for p in all_preds],
        "confidence":           all_confs,
    }).to_csv(csv_path, index=False)
    print(f"  CSV saved : {csv_path}")

    return {
        "dataset":           dataset_name,
        "total_images":      len(paths),
        "cracked_count":     n_cracked,
        "non_cracked_count": n_uncracked,
        "accuracy":          acc,
        "precision":         prec,
        "recall":            rec,
        "f1":                f1,
        "confusion_matrix":  cm,
        "report":            report,
        "all_preds":         all_preds,
        "all_labels":        all_labels_out,
        "all_paths":         all_paths_out,
        "all_confs":         all_confs,
        "time_s":            elapsed,
    }


# ============================================================================
# SECTION 8 — VISUALISATION HELPERS
# ============================================================================

def plot_confusion_matrix(cm, dataset_name, save_path):
    """Save a seaborn-styled confusion-matrix heatmap."""
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Cracked (0)", "Non-cracked (1)"],
                yticklabels=["Cracked (0)", "Non-cracked (1)"],
                ax=ax, annot_kws={"size": 16})
    ax.set_xlabel("Predicted Label", fontsize=13)
    ax.set_ylabel("True Label", fontsize=13)
    ax.set_title(f"Confusion Matrix — {dataset_name}\n"
                 f"(Cracked=0, Non-cracked=1)", fontsize=14)
    fig.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  CM plot   : {save_path}")


def plot_comparison_chart(results_list, save_path):
    """Grouped bar chart: Acc / Prec / Rec / F1 across all test datasets."""
    datasets = [r["dataset"] for r in results_list]
    metrics  = {
        "Accuracy":  [r["accuracy"]  for r in results_list],
        "Precision": [r["precision"] for r in results_list],
        "Recall":    [r["recall"]    for r in results_list],
        "F1-Score":  [r["f1"]        for r in results_list],
    }

    x = np.arange(len(datasets))
    width = 0.18
    colours = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

    fig, ax = plt.subplots(figsize=(14, 7))
    for i, (mname, vals) in enumerate(metrics.items()):
        bars = ax.bar(x + i * width, vals, width, label=mname,
                      color=colours[i], edgecolor="white", linewidth=0.5)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.008,
                    f"{v:.3f}", ha="center", va="bottom",
                    fontsize=8, fontweight="bold")

    ax.set_xlabel("Test Dataset", fontsize=13)
    ax.set_ylabel("Score", fontsize=13)
    ax.set_title("Cross-Dataset Generalization — DualBranchCrackNet\n"
                 "Metrics per Test Dataset  (pos_label=0 → Cracked = positive)",
                 fontsize=14)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(datasets, fontsize=10)
    ax.legend(loc="lower left", fontsize=11)
    ax.set_ylim(0, 1.18)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Chart     : {save_path}")


# ── Heatmap / Redline visualisation for individual images ──────────────────

def _single_image_heatmap(model, img_path):
    """
    Run a single image through the model and return:
        rgb_display (H,W,3 uint8)   – resized original (after IEAP)
        heatmap_np  (H,W float32)   – sigmoid heatmap [0-1]
    """
    bgr = cv2.imread(img_path)
    if bgr is None:
        return None, None
    bgr = cv2.resize(bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    bgr_enh = ieap_enhance(bgr)
    rgb = cv2.cvtColor(bgr_enh, cv2.COLOR_BGR2RGB)

    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    tensor = tfm(Image.fromarray(rgb)).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        _, heatmap, _ = model(tensor)
    heatmap_np = heatmap.squeeze().cpu().numpy()          # [256,256]
    return rgb, heatmap_np


def generate_vis_for_image(model, img_path, save_dir_heatmap,
                           save_dir_redline, dataset_tag,
                           true_label, pred_label, confidence):
    """
    Create:
      1)  3-panel figure  →  heatmaps/
      2)  standalone redline overlay  →  redline_visualizations/
    """
    rgb, heatmap_np = _single_image_heatmap(model, img_path)
    if rgb is None:
        return

    cat = ("TP" if (true_label == 0 and pred_label == 0)
           else "TN" if (true_label == 1 and pred_label == 1)
           else "FP" if (true_label == 1 and pred_label == 0)
           else "FN")

    hm_uint8  = (heatmap_np * 255).astype(np.uint8)
    hm_color  = cv2.applyColorMap(hm_uint8, cv2.COLORMAP_JET)
    hm_color  = cv2.cvtColor(hm_color, cv2.COLOR_BGR2RGB)
    overlay   = cv2.addWeighted(rgb, 0.55, hm_color, 0.45, 0)

    _, binary = cv2.threshold(hm_uint8, 0, 255,
                              cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    redline = rgb.copy()
    red_fill = redline.copy()
    red_fill[binary > 0] = [255, 50, 50]
    redline = cv2.addWeighted(redline, 0.6, red_fill, 0.4, 0)
    cv2.drawContours(redline, contours, -1, (255, 0, 0), 2)

    stem = os.path.splitext(os.path.basename(img_path))[0]

    # ── 1) 3-panel figure ──
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    true_name = LABEL_MAP[true_label]
    pred_name = LABEL_MAP[pred_label]

    axes[0].imshow(rgb)
    axes[0].set_title(f"Original Image\n"
                      f"True: {true_name} | Pred: {pred_name}\n"
                      f"Conf: {confidence:.4f}  [{cat}]", fontsize=10)
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title("Predicted Crack Localization\n"
                      "(Model Activation — NOT Ground Truth)", fontsize=10)
    axes[1].axis("off")

    axes[2].imshow(redline)
    axes[2].set_title("Redline Overlay\n"
                      "(Otsu-thresholded Model Activation)", fontsize=10)
    axes[2].axis("off")

    fig.suptitle(f"DualBranchCrackNet — {dataset_tag}  |  "
                 f"{os.path.basename(img_path)}", fontsize=11,
                 fontweight="bold")
    fig.tight_layout()
    os.makedirs(save_dir_heatmap, exist_ok=True)
    fig.savefig(os.path.join(save_dir_heatmap,
                             f"{dataset_tag}_{cat}_{stem}.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)

    # ── 2) standalone redline overlay ──
    os.makedirs(save_dir_redline, exist_ok=True)
    redline_bgr = cv2.cvtColor(redline, cv2.COLOR_RGB2BGR)
    cv2.imwrite(os.path.join(save_dir_redline,
                             f"{dataset_tag}_{cat}_{stem}_redline.png"),
                redline_bgr)


def generate_representative_examples(model, result, dataset_tag,
                                     heatmap_dir, redline_dir):
    """
    Find TP / TN / FP / FN examples and create visualisations for each.
    """
    print(f"\n  Generating heatmap / redline visualisations for {dataset_tag} …")

    preds  = result["all_preds"]
    labels = result["all_labels"]
    paths  = result["all_paths"]
    confs  = result["all_confs"]

    cats = {"TP": [], "TN": [], "FP": [], "FN": []}
    for i in range(len(preds)):
        t, p = labels[i], preds[i]
        if   t == 0 and p == 0: cats["TP"].append(i)
        elif t == 1 and p == 1: cats["TN"].append(i)
        elif t == 1 and p == 0: cats["FP"].append(i)
        elif t == 0 and p == 1: cats["FN"].append(i)

    for cat_name, indices in cats.items():
        if not indices:
            print(f"    {cat_name}: 0 examples available — skipping")
            continue

        n = min(NUM_VIS_EXAMPLES, len(indices))
        # Pick spread-out examples rather than just the first N
        step = max(1, len(indices) // (n + 1))
        sel  = [indices[min(step * (k + 1), len(indices) - 1)]
                for k in range(n)]

        for idx in sel:
            try:
                generate_vis_for_image(
                    model, paths[idx],
                    save_dir_heatmap=heatmap_dir,
                    save_dir_redline=redline_dir,
                    dataset_tag=dataset_tag,
                    true_label=labels[idx],
                    pred_label=preds[idx],
                    confidence=confs[idx],
                )
            except Exception as exc:
                print(f"    WARNING: vis failed for {paths[idx]}: {exc}")

        print(f"    {cat_name}: {n} visualisation(s) saved")


# ============================================================================
# SECTION 9 — SMOKE TEST
# ============================================================================

def smoke_test(model, sample_paths, sample_labels, output_base):
    """Quick sanity checks before the full experiment."""
    print(f"\n{'='*70}")
    print("SMOKE TEST  —  Quick Verification")
    print(f"{'='*70}")

    # 1) Forward-pass shapes
    print("  [1/4] Model forward pass …")
    with torch.no_grad():
        dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
        logits, hm, fused = model(dummy)
    assert logits.shape == (2, 2),          f"logits {logits.shape}"
    assert hm.shape     == (2, 1, 256, 256), f"heatmap {hm.shape}"
    assert fused.shape  == (2, 512, 16, 16), f"fused {fused.shape}"
    print("    ✓ Shapes OK:  logits=[2,2]  heatmap=[2,1,256,256]  "
          "fused=[2,512,16,16]")

    # 2) Real image
    print("  [2/4] Real image inference …")
    if len(sample_paths) >= 2:
        ds     = InferenceDataset(sample_paths[:2], sample_labels[:2], IMG_SIZE)
        loader = DataLoader(ds, batch_size=2, num_workers=0)
        with torch.no_grad():
            imgs, lbls, ps = next(iter(loader))
            logits, _, _ = model(imgs.to(DEVICE))
            probs = F.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
        for k in range(2):
            print(f"    img={os.path.basename(ps[k]):30s}  "
                  f"true={LABEL_MAP[lbls[k].item()]:12s}  "
                  f"pred={LABEL_MAP[preds[k]]:12s}  "
                  f"conf={probs[k].max():.4f}")
        print("    ✓ Real-image inference OK")
    else:
        print("    ⚠ Not enough sample images")

    # 3) Output directory
    print("  [3/4] Output directory write …")
    probe = os.path.join(output_base, "__probe__")
    os.makedirs(probe, exist_ok=True)
    with open(os.path.join(probe, "t.txt"), "w") as f:
        f.write("ok")
    os.remove(os.path.join(probe, "t.txt"))
    os.rmdir(probe)
    print("    ✓ Directory creation / write OK")

    # 4) CUDA
    print("  [4/4] Compute device …")
    if torch.cuda.is_available():
        gpu_props = torch.cuda.get_device_properties(0)
        vram = getattr(gpu_props, 'total_memory', getattr(gpu_props, 'total_mem', 0))
        print(f"    ✓ CUDA: {torch.cuda.get_device_name(0)}  "
              f"({vram / 1e9:.1f} GB)")
    else:
        print("    △ CPU only — functional but slower")

    print(f"\n  ✓✓✓ SMOKE TEST PASSED\n")


# ============================================================================
# SECTION 10 — COMPARISON TABLE GENERATION
# ============================================================================

def create_comparison_tables(results_list, output_base):
    """Write comparison_table.csv and comparison_table.md."""

    rows = []
    for r in results_list:
        rows.append({
            "Dataset":       r["dataset"],
            "Total_Images":  r["total_images"],
            "Cracked":       r["cracked_count"],
            "Non_Cracked":   r["non_cracked_count"],
            "Accuracy":      round(r["accuracy"],  4),
            "Precision":     round(r["precision"], 4),
            "Recall":        round(r["recall"],    4),
            "F1_Score":      round(r["f1"],        4),
            "Time_s":        round(r["time_s"],    1),
        })

    df = pd.DataFrame(rows)
    csv_path = os.path.join(output_base, "comparison_table.csv")
    df.to_csv(csv_path, index=False)
    print(f"  CSV : {csv_path}")

    # ── Markdown ──
    md = [
        "# Cross-Dataset Generalization Results — DualBranchCrackNet\n",
        "## Model",
        "- **Architecture**: DualBranchCrackNet (ResNet-50 + Swin-Tiny)",
        "- **Training data**: 70 % of Asphalt 720×720 (2,378 images)",
        "- **Label convention**: Cracked = 0, Non-cracked = 1",
        "- **Positive class**: Cracked (pos_label = 0)",
        "- **Checkpoint**: `crackdualnet_best.pth` (not retrained)\n",
        "## Results\n",
        "| Dataset | Images | Cracked | Non-Cracked | Accuracy | Precision | "
        "Recall | F1-Score |",
        "|---------|-------:|--------:|------------:|---------:|----------:|"
        "-------:|---------:|",
    ]
    for r in results_list:
        md.append(
            f"| {r['dataset']} | {r['total_images']:,} | "
            f"{r['cracked_count']:,} | {r['non_cracked_count']:,} | "
            f"{r['accuracy']:.4f} | {r['precision']:.4f} | "
            f"{r['recall']:.4f} | {r['f1']:.4f} |"
        )
    md += [
        "",
        "## Notes",
        "- Precision / Recall / F1 use `pos_label=0` (Cracked = positive).",
        "- No pixel-level IoU, Dice, or SSIM is reported — datasets lack "
        "pixel-level ground-truth masks.",
        "- Heatmaps = **Predicted Crack Localization (Model Activation)**, "
        "not ground-truth segmentation.",
        "- SUT-Crack and SDNET2018 have heavily imbalanced class distributions.",
        "",
        "## Ground-Truth Limitations",
        "All four datasets provide image-level binary labels only.",
        "Pixel-level ground-truth masks are NOT available; therefore IoU / Dice "
        "segmentation metrics are NOT applicable.",
        "",
        "## SSIM",
        "SSIM is not applicable — no valid reference image / mask is defined "
        "by this classification methodology.",
    ]

    md_path = os.path.join(output_base, "comparison_table.md")
    with open(md_path, "w") as f:
        f.write("\n".join(md) + "\n")
    print(f"  MD  : {md_path}")


# ============================================================================
# SECTION 11 — MAIN
# ============================================================================

def main():
    wall_t0 = time.time()

    print("=" * 70)
    print("CROSS-DATASET GENERALIZATION EXPERIMENT")
    print("DualBranchCrackNet — Pavement Crack Detection")
    print("=" * 70)
    print(f"  Device          : {DEVICE}")
    print(f"  Batch size      : {BATCH_SIZE}")
    print(f"  Image size      : {IMG_SIZE}")
    print(f"  Seed            : {SEED}")
    print(f"  Label convention: Cracked=0  Non-cracked=1")
    print(f"  Positive class  : Cracked (pos_label=0)")

    # ── 1. Detect paths ──────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("STEP 1 — PATH DETECTION")
    print(f"{'='*70}")
    paths = setup_paths()
    for k, v in paths.items():
        ok = "✓" if (v and os.path.exists(v)) else "✗"
        print(f"  {ok} {k:15s}: {v}")

    missing = [k for k, v in paths.items() if not v or not os.path.exists(v)]
    if missing:
        print(f"\n  ERROR: missing paths → {missing}")
        print("  Set MANUAL_* variables at the top of this script.")
        sys.exit(1)

    # Output directory
    script_dir  = os.path.dirname(os.path.abspath(__file__))
    output_base = os.path.join(script_dir, "results")
    if os.path.isdir("/kaggle/working"):
        output_base = "/kaggle/working/results"
    elif os.path.isdir("/content"):
        output_base = "/content/results"
    os.makedirs(output_base, exist_ok=True)
    print(f"\n  Output dir : {output_base}")

    # ── 2. Load model ────────────────────────────────────────────────────
    model = load_model(paths["checkpoint"])

    # ── 3. Load datasets ─────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("STEP 3 — LOADING DATASETS")
    print(f"{'='*70}")

    # 3a  Asphalt 720 — 70/30 split
    print("\n  [1/4] Asphalt 720×720  (70/30 stratified split):")
    (train_p720, test_p720,
     train_l720, test_l720, cmap720) = load_asphalt720_split(paths["asphalt_720"])
    print(f"    Train : {len(train_p720)}  "
          f"(C={sum(1 for l in train_l720 if l==0)}, "
          f"NC={sum(1 for l in train_l720 if l==1)})")
    print(f"    Test  : {len(test_p720)}  "
          f"(C={sum(1 for l in test_l720 if l==0)}, "
          f"NC={sum(1 for l in test_l720 if l==1)})")

    # Save split manifest
    split_csv = os.path.join(output_base, "asphalt720_train_test_split.csv")
    pd.DataFrame({
        "path":  train_p720 + test_p720,
        "label": train_l720 + test_l720,
        "split": ["train"] * len(train_p720) + ["test"] * len(test_p720),
    }).to_csv(split_csv, index=False)
    print(f"    Split manifest → {split_csv}")

    # 3b  Asphalt 360
    print("\n  [2/4] Asphalt 360×360  (full):")
    a360_p, a360_l, a360_c, a360_u = load_labeled_dataset(
        paths["asphalt_360"], "Cracked360", "Uncracked360")
    print(f"    Total={len(a360_p)}  C={a360_c}  NC={a360_u}")

    # 3c  SUT-Crack
    print("\n  [3/4] SUT-Crack Classification  (full):")
    sut_p, sut_l, sut_c, sut_u = load_labeled_dataset(
        paths["sut_crack"], "With Crack", "Without Crack")
    print(f"    Total={len(sut_p)}  C={sut_c}  NC={sut_u}")

    # 3d  SDNET2018
    print("\n  [4/4] SDNET2018 Pavements  (full):")
    sd_p, sd_l, sd_c, sd_u = load_labeled_dataset(
        paths["sdnet"], "Cracked", "Non-cracked")
    print(f"    Total={len(sd_p)}  C={sd_c}  NC={sd_u}")

    # ── 4. Leakage checks ────────────────────────────────────────────────
    test_datasets_map = {
        "Asphalt720_HeldOut": test_p720,
        "Asphalt360":         a360_p,
        "SUT_Crack":          sut_p,
        "SDNET2018":          sd_p,
    }
    check_leakage(train_p720, test_datasets_map)

    # ── 5. Smoke test ────────────────────────────────────────────────────
    smoke_test(model, test_p720, test_l720, output_base)

    # ── 6. Full evaluation ───────────────────────────────────────────────
    eval_queue = [
        ("Asphalt720_HeldOut30pct", test_p720, test_l720, "asphalt720"),
        ("Asphalt360_Full",         a360_p,    a360_l,    "asphalt360"),
        ("SUT_Crack_Full",          sut_p,     sut_l,     "sut_crack"),
        ("SDNET2018_Full",          sd_p,      sd_l,      "sdnet2018"),
    ]

    all_results = []
    heatmap_dir = os.path.join(output_base, "heatmaps")
    redline_dir = os.path.join(output_base, "redline_visualizations")
    cm_dir      = os.path.join(output_base, "confusion_matrices")

    for ds_name, ds_p, ds_l, folder in eval_queue:
        ds_out = os.path.join(output_base, folder)
        result = evaluate_dataset(model, ds_p, ds_l, ds_name, ds_out)
        all_results.append(result)

        # Confusion-matrix plot
        plot_confusion_matrix(
            result["confusion_matrix"], ds_name,
            os.path.join(cm_dir, f"{folder}_confusion_matrix.png"))

        # Representative visualisations
        generate_representative_examples(
            model, result, folder, heatmap_dir, redline_dir)

        # Free GPU memory between datasets
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    # ── 7. Comparison outputs ────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("GENERATING COMPARISON OUTPUTS")
    print(f"{'='*70}")

    create_comparison_tables(all_results, output_base)

    plot_comparison_chart(
        all_results,
        os.path.join(output_base, "metrics_comparison_chart.png"))

    # ── 8. Final summary ─────────────────────────────────────────────────
    wall_elapsed = time.time() - wall_t0

    print(f"\n{'='*70}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*70}")
    print(f"  Total wall-clock time : {wall_elapsed:.1f}s "
          f"({wall_elapsed/60:.1f} min)")
    print(f"  Results directory     : {output_base}\n")

    print("  Output tree:")
    for root, dirs, files in sorted(os.walk(output_base)):
        level  = root.replace(output_base, "").count(os.sep)
        indent = "    " + "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        sub    = "    " + "  " * (level + 1)
        for fn in sorted(files):
            print(f"{sub}{fn}")

    print(f"\n  {'─'*60}")
    print("  CONFIRMATIONS")
    print(f"  {'─'*60}")
    print("  ✓ Checkpoint NOT modified (read-only evaluation)")
    print("  ✓ No retraining performed")
    print("  ✓ No fabricated ground-truth masks or metrics")
    print("  ✓ No pixel-level IoU / Dice / SSIM reported")
    print("  ✓ Heatmaps labelled 'Predicted Crack Localization "
          "(Model Activation)'")
    print("  ✓ Label convention preserved: Cracked=0, Non-cracked=1")
    print("  ✓ Positive class for metrics: Cracked (pos_label=0)")
    print("  ✓ Each dataset evaluated independently — no merging")
    print("  ✓ Data leakage checks performed and passed")
    print(f"  {'─'*60}")


# ============================================================================
if __name__ == "__main__":
    main()



Overwriting experiment_cross_dataset.py


In [5]:
!python experiment_cross_dataset.py


CROSS-DATASET GENERALIZATION EXPERIMENT
DualBranchCrackNet — Pavement Crack Detection
  Device          : cuda
  Batch size      : 64
  Image size      : 256
  Seed            : 42
  Label convention: Cracked=0  Non-cracked=1
  Positive class  : Cracked (pos_label=0)

STEP 1 — PATH DETECTION
  ✓ checkpoint     : /kaggle/input/models/dinjbu/crackdualnet/pytorch/default/1/crackdualnet_best.pth
  ✓ asphalt_720    : /kaggle/input/datasets/dinjbu/asphalt-cracked-and-uncracked-image-dataset-720/Cracked and Uncracked Image Database_720x720
  ✓ asphalt_360    : /kaggle/input/datasets/dinjbu/asphalt-360/Cracked and Uncracked Image Database_360x360
  ✓ sut_crack      : /kaggle/input/datasets/dinjbu/sut-crack/3-Classification
  ✓ sdnet          : /kaggle/input/datasets/dinjbu/sdnet-pavement-2018/SDNET /Pavements

  Output dir : /kaggle/working/results

LOADING MODEL
  Checkpoint : /kaggle/input/models/dinjbu/crackdualnet/pytorch/default/1/crackdualnet_best.pth
  File size  : 86.2 MB
  Device     

In [5]:
%%writefile experiment_pseudo_segmentation.py
#!/usr/bin/env python3
"""
==============================================================================
PSEUDO-GROUND-TRUTH & GRAD-CAM LOCALIZATION EVALUATION PIPELINE
DualBranchCrackNet — Pavement Crack Detection
==============================================================================
"""

import os
import sys
import glob
import math
import hashlib
import time
import argparse
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import timm

from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize
from skimage.metrics import structural_similarity

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

# ============================================================================
# CONSTANTS & CONFIGURATION
# ============================================================================

EXPECTED_MD5 = "6a005166a7147906bfac621018b378da"
IMG_SIZE = 256
NUM_CLASSES = 2
SEED = 42
BATCH_SIZE = 16  # Reduced batch size for GPU VRAM safety

# Label convention: Cracked=0, Non-cracked=1
CLASS_NAMES = {0: "Cracked", 1: "Non-cracked"}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR = os.path.join("results", "pseudo_gt_gradcam_evaluation")
SUBDIRS = [
    "pseudo_gt_masks",
    "gradcam_masks",
    "gradcam_overlays",
    "redline_visualizations",
    "comparison_visualizations",
    "severity_visualizations",
    "csv",
    "metrics",
    "documentation",
]

# ============================================================================
# UTILITIES & VERIFICATION
# ============================================================================

def compute_md5(file_path):
    hasher = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def verify_checkpoint(ckpt_path):
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found at: {ckpt_path}")
    md5_val = compute_md5(ckpt_path)
    print(f"  Checkpoint MD5 : {md5_val}", flush=True)
    if md5_val != EXPECTED_MD5:
        raise ValueError(
            f"MD5 mismatch! Expected {EXPECTED_MD5}, got {md5_val}. "
            "Checkpoint may have been modified or corrupted!"
        )
    print("  ✓ Checkpoint MD5 verified successfully.", flush=True)

def ensure_directories(base_dir):
    os.makedirs(base_dir, exist_ok=True)
    dirs = {}
    for sd in SUBDIRS:
        p = os.path.join(base_dir, sd)
        os.makedirs(p, exist_ok=True)
        dirs[sd] = p
    return dirs

# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

class DualBranchCrackNet(nn.Module):
    def __init__(self, num_classes=2, fused_channels=512, fused_size=16, img_size=256):
        super().__init__()
        self.fused_size = fused_size

        resnet = models.resnet50(weights=None)
        self.cnn_stem = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3,
        )
        cnn_out_channels = 1024

        self.swin = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=False,
            features_only=True,
            out_indices=(2,),
            img_size=img_size,
        )
        swin_out_channels = self.swin.feature_info.channels()[0]

        self.cnn_proj  = nn.Conv2d(cnn_out_channels, fused_channels, 1)
        self.swin_proj = nn.Conv2d(swin_out_channels, fused_channels, 1)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(fused_channels, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

        self.heatmap_conv = nn.Conv2d(fused_channels, 1, kernel_size=1)

    def forward(self, x):
        f_cnn = self.cnn_proj(self.cnn_stem(x))
        f_cnn = F.interpolate(f_cnn, size=(self.fused_size, self.fused_size),
                              mode="bilinear", align_corners=False)

        f_swin = self.swin(x)[0]
        if f_swin.shape[1] != self.swin_proj.in_channels:
            f_swin = f_swin.permute(0, 3, 1, 2).contiguous()
        f_swin = self.swin_proj(f_swin)
        f_swin = F.interpolate(f_swin, size=(self.fused_size, self.fused_size),
                               mode="bilinear", align_corners=False)

        fused = f_cnn + f_swin

        logits = self.classifier(fused)
        heatmap = torch.sigmoid(self.heatmap_conv(fused))
        heatmap_full = F.interpolate(heatmap, size=(256, 256),
                                     mode="bilinear", align_corners=False)
        return logits, heatmap_full, fused

def load_model(checkpoint_path):
    verify_checkpoint(checkpoint_path)
    model = DualBranchCrackNet(num_classes=NUM_CLASSES, img_size=IMG_SIZE)
    state_dict = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict, strict=True)
    model.to(DEVICE)
    model.eval()
    return model

# ============================================================================
# IMAGE PREPROCESSING & DATASET LOADER
# ============================================================================

def preprocess_single_image(img_bgr):
    img_resized = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    lab = cv2.cvtColor(img_resized, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe_bgr = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR)

    img_rgb = cv2.cvtColor(img_clahe_bgr, cv2.COLOR_BGR2RGB)

    img_float = img_rgb.astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    norm_img = (img_float - mean) / std

    tensor = torch.from_numpy(norm_img.transpose(2, 0, 1)).float()
    return img_resized, tensor

class ImageEvalDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            img_bgr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img_resized, tensor = preprocess_single_image(img_bgr)
        return tensor, img_resized, path, label

# ============================================================================
# BATCHED GRAD-CAM IMPLEMENTATION WITH VRAM CLEANUP
# ============================================================================

def generate_gradcam_masks_batch(model, tensor_batch, target_class=0):
    tensor_batch = tensor_batch.to(DEVICE).requires_grad_(True)
    model.zero_grad()

    fused_gradients = None
    def backward_hook(grad):
        nonlocal fused_gradients
        fused_gradients = grad

    logits, _, fused = model(tensor_batch)
    h_hook = fused.register_hook(backward_hook)

    scores = logits[:, target_class]
    scores.sum().backward()
    h_hook.remove()

    probs = F.softmax(logits.detach(), dim=1).cpu().numpy()
    pred_classes = np.argmax(probs, axis=1)
    confidences = probs[np.arange(len(probs)), pred_classes]

    grads_all = fused_gradients.detach().cpu().numpy()
    acts_all = fused.detach().cpu().numpy()

    results = []
    batch_size = len(tensor_batch)
    for b in range(batch_size):
        grads = grads_all[b]
        acts = acts_all[b]

        weights = np.mean(grads, axis=(1, 2))
        cam = np.zeros(acts.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * acts[i]

        cam = np.maximum(cam, 0)

        cam_min, cam_max = cam.min(), cam.max()
        if cam_max > cam_min:
            cam_norm = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam_norm = np.zeros_like(cam)

        cam_resized = cv2.resize(cam_norm, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
        cam_resized = np.clip(cam_resized, 0.0, 1.0)
        cam_uint8 = np.uint8(255 * cam_resized)

        _, binary_mask = cv2.threshold(cam_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)

        results.append((cam_resized, binary_mask, int(pred_classes[b]), float(confidences[b])))

    del tensor_batch, logits, fused, fused_gradients
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

# ============================================================================
# PSEUDO-GROUND-TRUTH / REFERENCE MASK GENERATOR
# ============================================================================

def generate_pseudo_gt_reference_mask(img_bgr_256):
    gray = cv2.cvtColor(img_bgr_256, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    denoised = cv2.bilateralFilter(enhanced, d=7, sigmaColor=50, sigmaSpace=50)

    kernel_tophat = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    tophat = cv2.morphologyEx(denoised, cv2.MORPH_BLACKHAT, kernel_tophat)

    _, thresh = cv2.threshold(tophat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel_clean = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_clean)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(cleaned, connectivity=8)
    ref_mask = np.zeros_like(cleaned)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= 20:
            ref_mask[labels == i] = 255

    return ref_mask

# ============================================================================
# MASK COMPARISON METRICS
# ============================================================================

def compute_mask_metrics(ref_mask, pred_mask):
    P = pred_mask > 127
    R = ref_mask > 127
    ref_pixels = int(R.sum())
    pred_pixels = int(P.sum())

    if ref_pixels == 0 and pred_pixels == 0:
        pseudo_iou = 1.0
        pseudo_dice = 1.0
        ssim_val = 1.0
    elif ref_pixels == 0 or pred_pixels == 0:
        pseudo_iou = 0.0
        pseudo_dice = 0.0
        ssim_val = float(structural_similarity(ref_mask, pred_mask, data_range=255))
    else:
        intersection = int(np.logical_and(P, R).sum())
        union = int(np.logical_or(P, R).sum())
        pseudo_iou = float(intersection / union) if union > 0 else 1.0
        pseudo_dice = float((2.0 * intersection) / (pred_pixels + ref_pixels)) if (pred_pixels + ref_pixels) > 0 else 1.0
        ssim_val = float(structural_similarity(ref_mask, pred_mask, data_range=255))

    return {
        "pseudo_gt_pixel_count": ref_pixels,
        "predicted_mask_pixel_count": pred_pixels,
        "pseudo_iou": pseudo_iou,
        "pseudo_dice": pseudo_dice,
        "ssim": ssim_val,
    }

# ============================================================================
# CRACK CHARACTERISTICS & RULE-BASED SEVERITY ESTIMATE
# ============================================================================

def analyze_crack_mask(pred_mask):
    P = pred_mask > 127
    total_pixels = IMG_SIZE * IMG_SIZE
    pred_pixels = int(P.sum())
    crack_density = float(pred_pixels / total_pixels)

    if pred_pixels == 0:
        return {
            "crack_density": 0.0,
            "crack_length": 0.0,
            "crack_width": 0.0,
            "severity": "Low",
        }

    skel = skeletonize(P)
    crack_length = float(skel.sum())

    edt = distance_transform_edt(P)
    if crack_length > 0:
        crack_width = float(2.0 * np.mean(edt[skel]))
    else:
        crack_width = 0.0

    if crack_density >= 0.03 or crack_length >= 150.0:
        severity = "High"
    elif crack_density >= 0.01 or crack_length >= 50.0:
        severity = "Medium"
    else:
        severity = "Low"

    return {
        "crack_density": crack_density,
        "crack_length": crack_length,
        "crack_width": crack_width,
        "severity": severity,
    }

# ============================================================================
# PARALLEL WORKER FUNCTION
# ============================================================================

def process_image_worker(item):
    img_p, true_cls, ds_display_name, pred_cls, conf, cam_resized, pred_mask, img_bgr_256 = item

    ref_mask = generate_pseudo_gt_reference_mask(img_bgr_256)
    comp_metrics = compute_mask_metrics(ref_mask, pred_mask)
    crack_info = analyze_crack_mask(pred_mask)

    record = {
        "dataset": ds_display_name,
        "image_path": img_p,
        "true_class": CLASS_NAMES[true_cls],
        "predicted_class": CLASS_NAMES[pred_cls],
        "confidence": conf,
        "pseudo_gt_pixel_count": comp_metrics["pseudo_gt_pixel_count"],
        "predicted_mask_pixel_count": comp_metrics["predicted_mask_pixel_count"],
        "pseudo_iou": comp_metrics["pseudo_iou"],
        "pseudo_dice": comp_metrics["pseudo_dice"],
        "ssim": comp_metrics["ssim"],
        "crack_density": crack_info["crack_density"],
        "crack_length": crack_info["crack_length"],
        "crack_width": crack_info["crack_width"],
        "severity": crack_info["severity"],
    }
    return record, ref_mask, crack_info

# ============================================================================
# DATASET DISCOVERY & SPLITS
# ============================================================================

def collect_images(folder):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.tif", "*.tiff")
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(folder, "**", ext), recursive=True))
        paths.extend(glob.glob(os.path.join(folder, "**", ext.upper()), recursive=True))
    return sorted(list(set(paths)))

def discover_datasets(search_roots):
    print(f"\n{'='*70}", flush=True)
    print("PATH DISCOVERY", flush=True)
    print(f"{'='*70}", flush=True)

    def find_dir(roots, target_subdirs):
        for root in roots:
            if not os.path.isdir(root):
                continue
            for dirpath, dirnames, _ in os.walk(root):
                if "1-segmentation" in dirpath.lower():
                    continue
                children_lower = [d.lower() for d in dirnames]
                if all(t.lower() in children_lower for t in target_subdirs):
                    return os.path.abspath(dirpath)
        return None

    ckpt_candidates = [
        "training/crackdualnet_best.pth",
        "/kaggle/input/models/dinjbu/crackdualnet/pytorch/default/1/crackdualnet_best.pth",
        "crackdualnet_best.pth"
    ]
    ckpt = None
    for c in ckpt_candidates:
        if os.path.exists(c):
            ckpt = os.path.abspath(c)
            break

    a720 = find_dir(search_roots, ["Cracked720", "Uncracked720"])
    a360 = find_dir(search_roots, ["Cracked360", "Uncracked360"])

    sut = None
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            if "1-segmentation" in dirpath.lower():
                continue
            if os.path.basename(dirpath) in ["Classification", "3-Classification"]:
                children_lower = [d.lower() for d in dirnames]
                if "with crack" in children_lower and "without crack" in children_lower:
                    sut = os.path.abspath(dirpath)
                    break
        if sut:
            break

    sdnet = None
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            if "720" in dirpath or "360" in dirpath or "1-segmentation" in dirpath.lower():
                continue
            children_lower = [d.lower() for d in dirnames]
            if "cracked" in children_lower and "non-cracked" in children_lower:
                sdnet = os.path.abspath(dirpath)
                break
        if sdnet:
            break

    print(f"  ✓ checkpoint  : {ckpt}", flush=True)
    print(f"  ✓ asphalt_720 : {a720}", flush=True)
    print(f"  ✓ asphalt_360 : {a360}", flush=True)
    print(f"  ✓ sut_crack   : {sut}", flush=True)
    print(f"  ✓ sdnet       : {sdnet}", flush=True)

    return {
        "checkpoint": ckpt,
        "asphalt_720": a720,
        "asphalt_360": a360,
        "sut_crack": sut,
        "sdnet": sdnet,
    }

def get_dataset_files(dataset_key, path):
    if dataset_key == "asphalt_720":
        cracked = collect_images(os.path.join(path, "Cracked720"))
        uncracked = collect_images(os.path.join(path, "Uncracked720"))
        all_paths = cracked + uncracked
        all_labels = [0] * len(cracked) + [1] * len(uncracked)
        train_p, test_p, train_l, test_l = train_test_split(
            all_paths, all_labels, test_size=0.30, random_state=SEED, stratify=all_labels
        )
        return test_p, test_l
    elif dataset_key == "asphalt_360":
        cracked = collect_images(os.path.join(path, "Cracked360"))
        uncracked = collect_images(os.path.join(path, "Uncracked360"))
        return cracked + uncracked, [0] * len(cracked) + [1] * len(uncracked)
    elif dataset_key == "sut_crack":
        cracked = collect_images(os.path.join(path, "With Crack"))
        uncracked = collect_images(os.path.join(path, "Without Crack"))
        return cracked + uncracked, [0] * len(cracked) + [1] * len(uncracked)
    elif dataset_key == "sdnet":
        cracked = collect_images(os.path.join(path, "Cracked"))
        uncracked = collect_images(os.path.join(path, "Non-cracked"))
        return cracked + uncracked, [0] * len(cracked) + [1] * len(uncracked)
    else:
        raise ValueError(f"Unknown dataset key: {dataset_key}")

# ============================================================================
# VISUALIZATION GENERATORS
# ============================================================================

def save_sample_visualizations(img_bgr_256, cam_resized, pred_mask, ref_mask, crack_info,
                               dataset_name, img_idx, dir_map):
    img_rgb = cv2.cvtColor(img_bgr_256, cv2.COLOR_BGR2RGB)

    heatmap_colored = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(img_bgr_256, 0.6, heatmap_colored, 0.4, 0)

    redline_img = img_rgb.copy()
    contours, _ = cv2.findContours(pred_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(redline_img, contours, -1, (255, 0, 0), 2)

    fname = f"{dataset_name}_sample_{img_idx:04d}.png"

    cv2.imwrite(os.path.join(dir_map["pseudo_gt_masks"], fname), ref_mask)
    cv2.imwrite(os.path.join(dir_map["gradcam_masks"], fname), pred_mask)
    cv2.imwrite(os.path.join(dir_map["gradcam_overlays"], fname), overlay)
    cv2.imwrite(os.path.join(dir_map["redline_visualizations"], fname), cv2.cvtColor(redline_img, cv2.COLOR_RGB2BGR))

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img_rgb)
    axes[0].set_title("Original Image", fontsize=11, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(ref_mask, cmap="gray")
    axes[1].set_title("Pseudo-GT / Reference Mask", fontsize=11, fontweight="bold")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Grad-CAM Predicted Mask", fontsize=11, fontweight="bold")
    axes[2].axis("off")

    comp = np.zeros((256, 256, 3), dtype=np.uint8)
    comp[ref_mask > 127, 1] = 255
    comp[pred_mask > 127, 0] = 255
    axes[3].imshow(comp)
    axes[3].set_title("Overlay Comparison\n(Green:Ref, Red:Pred, Yellow:Overlap)", fontsize=9, fontweight="bold")
    axes[3].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(dir_map["comparison_visualizations"], fname), dpi=150, bbox_inches="tight")
    plt.close()

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img_rgb)
    axes[0].set_title("1. Original Image", fontsize=11, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[1].set_title("2. Grad-CAM Heatmap", fontsize=11, fontweight="bold")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("3. Predicted Crack Mask", fontsize=11, fontweight="bold")
    axes[2].axis("off")

    axes[3].imshow(redline_img)
    axes[3].set_title(f"4. Red Localization\n({crack_info['severity']} Severity)", fontsize=11, fontweight="bold")
    axes[3].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(dir_map["severity_visualizations"], fname), dpi=150, bbox_inches="tight")
    plt.close()

# ============================================================================
# MAIN EVALUATION PIPELINE
# ============================================================================

def run_evaluation():
    print(f"\n{'='*70}", flush=True)
    print("STARTING PSEUDO-GT & GRAD-CAM EVALUATION PIPELINE", flush=True)
    print(f"{'='*70}", flush=True)

    search_roots = [
        ".", "..",
        "/kaggle/input", "/kaggle/input/datasets/dinjbu",
        os.path.expanduser("~/Desktop/TARP_FINAL")
    ]
    discovered = discover_datasets(search_roots)

    ckpt_path = discovered["checkpoint"]
    if not ckpt_path:
        raise FileNotFoundError("Could not locate crackdualnet_best.pth!")

    verify_checkpoint(ckpt_path)
    model = load_model(ckpt_path)
    dir_map = ensure_directories(OUTPUT_DIR)

    dataset_keys = [
        ("Asphalt 720 (Held-out 30%)", "asphalt_720"),
        ("Asphalt 360 (Full)", "asphalt_360"),
        ("SUT-Crack Classification (Full)", "sut_crack"),
        ("SDNET2018 (Full)", "sdnet"),
    ]

    all_pred_records = []
    dataset_metrics_records = []
    severity_summary_records = []

    num_workers = max(1, os.cpu_count() - 1)
    print(f"  Initialized Parallel Processing Pool with {num_workers} CPU Workers.", flush=True)

    for ds_display_name, ds_key in dataset_keys:
        ds_path = discovered[ds_key]
        if not ds_path or not os.path.exists(ds_path):
            print(f"\n[WARNING] Dataset path for {ds_display_name} not found! Skipping...", flush=True)
            continue

        paths, labels = get_dataset_files(ds_key, ds_path)
        num_images = len(paths)
        print(f"\n{'='*70}", flush=True)
        print(f"EVALUATING DATASET: {ds_display_name}", flush=True)
        print(f"  Path        : {ds_path}", flush=True)
        print(f"  Total Images: {num_images}", flush=True)
        print(f"{'='*70}", flush=True)

        ds_records = []
        sample_saved_count = 0
        max_samples_to_save = 15

        eval_dataset = ImageEvalDataset(paths, labels)
        loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=False)

        items_to_process = []
        start_time = time.time()
        processed_count = 0

        for tensor_batch, img_bgr_batch, path_batch, label_batch in loader:
            batch_results = generate_gradcam_masks_batch(
                model, tensor_batch, target_class=0
            )

            for b in range(len(batch_results)):
                cam_resized, pred_mask, pred_cls, conf = batch_results[b]
                img_p = path_batch[b]
                true_cls = int(label_batch[b])
                img_bgr_256 = img_bgr_batch[b].numpy()

                if sample_saved_count < max_samples_to_save and (true_cls == 0 or pred_mask.sum() > 500):
                    ref_mask_sample = generate_pseudo_gt_reference_mask(img_bgr_256)
                    crack_info_sample = analyze_crack_mask(pred_mask)
                    save_sample_visualizations(
                        img_bgr_256, cam_resized, pred_mask, ref_mask_sample, crack_info_sample,
                        ds_key, sample_saved_count, dir_map
                    )
                    sample_saved_count += 1

                items_to_process.append((
                    img_p, true_cls, ds_display_name, pred_cls, conf, cam_resized, pred_mask, img_bgr_256
                ))

            processed_count += len(batch_results)
            if processed_count % 1000 < BATCH_SIZE or processed_count == num_images:
                elapsed = time.time() - start_time
                rate = processed_count / elapsed if elapsed > 0 else 0
                print(f"  [Model Grad-CAM Inference] {processed_count}/{num_images} images [{elapsed:.1f}s, {rate:.1f} img/s]...", flush=True)

        print(f"  [Parallel Metrics Calculation] Processing {len(items_to_process)} images across {num_workers} workers...", flush=True)
        t_par_start = time.time()

        with ProcessPoolExecutor(max_workers=num_workers) as executor:
            futures = [executor.submit(process_image_worker, item) for item in items_to_process]
            done_count = 0
            for future in as_completed(futures):
                record, ref_mask, crack_info = future.result()
                ds_records.append(record)
                all_pred_records.append(record)

                done_count += 1
                if done_count % 5000 == 0 or done_count == len(items_to_process):
                    elapsed_par = time.time() - t_par_start
                    print(f"    Completed {done_count}/{len(items_to_process)} image metrics [{elapsed_par:.1f}s]...", flush=True)

        df_ds = pd.DataFrame(ds_records)

        iou_all_mean = df_ds["pseudo_iou"].mean()
        iou_all_med = df_ds["pseudo_iou"].median()
        dice_all_mean = df_ds["pseudo_dice"].mean()
        dice_all_med = df_ds["pseudo_dice"].median()
        ssim_all_mean = df_ds["ssim"].mean()
        ssim_all_med = df_ds["ssim"].median()

        df_nonempty = df_ds[df_ds["pseudo_gt_pixel_count"] > 0]
        num_nonempty = len(df_nonempty)

        if num_nonempty > 0:
            iou_ne_mean = df_nonempty["pseudo_iou"].mean()
            iou_ne_med = df_nonempty["pseudo_iou"].median()
            dice_ne_mean = df_nonempty["pseudo_dice"].mean()
            dice_ne_med = df_nonempty["pseudo_dice"].median()
            ssim_ne_mean = df_nonempty["ssim"].mean()
            ssim_ne_med = df_nonempty["ssim"].median()
        else:
            iou_ne_mean = iou_ne_med = dice_ne_mean = dice_ne_med = ssim_ne_mean = ssim_ne_med = 0.0

        dataset_metrics_records.append({
            "dataset": ds_display_name,
            "image_count": num_images,
            "non_empty_ref_count": num_nonempty,
            "mean_pseudo_iou_all": iou_all_mean,
            "median_pseudo_iou_all": iou_all_med,
            "mean_pseudo_iou_nonempty": iou_ne_mean,
            "median_pseudo_iou_nonempty": iou_ne_med,
            "mean_pseudo_dice_all": dice_all_mean,
            "median_pseudo_dice_all": dice_all_med,
            "mean_pseudo_dice_nonempty": dice_ne_mean,
            "median_pseudo_dice_nonempty": dice_ne_med,
            "mean_ssim_all": ssim_all_mean,
            "median_ssim_all": ssim_all_med,
            "mean_ssim_nonempty": ssim_ne_mean,
            "median_ssim_nonempty": ssim_ne_med,
        })

        sev_counts = df_ds["severity"].value_counts().to_dict()
        severity_summary_records.append({
            "dataset": ds_display_name,
            "Low": sev_counts.get("Low", 0),
            "Medium": sev_counts.get("Medium", 0),
            "High": sev_counts.get("High", 0),
        })

        print(f"\n  RESULTS FOR {ds_display_name}:", flush=True)
        print(f"    Pseudo-IoU  (All)       : Mean={iou_all_mean:.4f}, Median={iou_all_med:.4f}", flush=True)
        print(f"    Pseudo-IoU  (Non-Empty) : Mean={iou_ne_mean:.4f}, Median={iou_ne_med:.4f}", flush=True)
        print(f"    Pseudo-Dice (All)       : Mean={dice_all_mean:.4f}, Median={dice_all_med:.4f}", flush=True)
        print(f"    Pseudo-Dice (Non-Empty) : Mean={dice_ne_mean:.4f}, Median={dice_ne_med:.4f}", flush=True)
        print(f"    SSIM        (All)       : Mean={ssim_all_mean:.4f}, Median={ssim_all_med:.4f}", flush=True)
        print(f"    SSIM        (Non-Empty) : Mean={ssim_ne_mean:.4f}, Median={ssim_ne_med:.4f}", flush=True)

    print(f"\n{'='*70}", flush=True)
    print("SAVING CSV REPORTS & GENERATING COMPARISON PLOTS", flush=True)
    print(f"{'='*70}", flush=True)

    df_all_pred = pd.DataFrame(all_pred_records)
    csv_preds_path = os.path.join(dir_map["csv"], "pseudo_gt_predictions.csv")
    df_all_pred.to_csv(csv_preds_path, index=False)
    print(f"  ✓ Saved: {csv_preds_path}", flush=True)

    df_loc_metrics = pd.DataFrame(dataset_metrics_records)
    csv_loc_path = os.path.join(dir_map["csv"], "dataset_localization_metrics.csv")
    df_loc_metrics.to_csv(csv_loc_path, index=False)
    print(f"  ✓ Saved: {csv_loc_path}", flush=True)

    df_sev_summary = pd.DataFrame(severity_summary_records)
    csv_sev_path = os.path.join(dir_map["csv"], "severity_summary.csv")
    df_sev_summary.to_csv(csv_sev_path, index=False)
    print(f"  ✓ Saved: {csv_sev_path}", flush=True)

    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(df_loc_metrics))
    width = 0.35
    ax.bar(x - width/2, df_loc_metrics["mean_pseudo_iou_all"], width, label="Pseudo-IoU (All)", color="#2b5c8f")
    ax.bar(x + width/2, df_loc_metrics["mean_pseudo_iou_nonempty"], width, label="Pseudo-IoU (Non-Empty)", color="#4682b4")
    ax.set_xticks(x)
    ax.set_xticklabels([d.split()[0] for d in df_loc_metrics["dataset"]], rotation=15)
    ax.set_ylabel("Mean Pseudo-IoU", fontsize=11, fontweight="bold")
    ax.set_title("Dataset-Wise Pseudo-IoU Comparison", fontsize=13, fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(dir_map["metrics"], "pseudo_iou_by_dataset.png"), dpi=200)
    plt.close()

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x - width/2, df_loc_metrics["mean_ssim_all"], width, label="SSIM (All Images)", color="#2e8b57")
    ax.bar(x + width/2, df_loc_metrics["mean_ssim_nonempty"], width, label="SSIM (Non-Empty Ref)", color="#3cb371")
    ax.set_xticks(x)
    ax.set_xticklabels([d.split()[0] for d in df_loc_metrics["dataset"]], rotation=15)
    ax.set_ylabel("Mean SSIM", fontsize=11, fontweight="bold")
    ax.set_title("Dataset-Wise Structural Similarity (SSIM)", fontsize=13, fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(dir_map["metrics"], "ssim_by_dataset.png"), dpi=200)
    plt.close()

    fig, ax = plt.subplots(figsize=(10, 6))
    df_sev_summary.set_index("dataset")[["Low", "Medium", "High"]].plot(kind="bar", stacked=True, ax=ax, color=["#2ecc71", "#f39c12", "#e74c3c"])
    ax.set_ylabel("Image Count", fontsize=11, fontweight="bold")
    ax.set_title("Rule-Based Crack Severity Distribution Across Datasets", fontsize=13, fontweight="bold")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(os.path.join(dir_map["metrics"], "severity_distribution.png"), dpi=200)
    plt.close()

    print(f"\n{'='*70}", flush=True)
    print("FINAL CHECKPOINT INTEGRITY VERIFICATION", flush=True)
    print(f"{'='*70}", flush=True)
    verify_checkpoint(ckpt_path)

    generate_readme(OUTPUT_DIR, df_loc_metrics, df_sev_summary)
    print(f"\n  ✓ Generated documentation: {os.path.join(OUTPUT_DIR, 'README.md')}", flush=True)

    print(f"\n{'='*70}", flush=True)
    print("EVALUATION PIPELINE COMPLETED SUCCESSFULLY", flush=True)
    print(f"{'='*70}", flush=True)

# ============================================================================
# README / RESEARCH DOCUMENTATION GENERATOR
# ============================================================================

def generate_readme(output_base_dir, df_metrics, df_severity):
    readme_path = os.path.join(output_base_dir, "README.md")

    metrics_markdown = df_metrics.to_markdown(index=False)
    severity_markdown = df_severity.to_markdown(index=False)

    readme_content = f"""# Pseudo-Ground-Truth & Grad-CAM Localization Evaluation Report

## Mandatory Research Disclaimer
> [!IMPORTANT]
> **The evaluated datasets do not provide pixel-level crack annotations. Therefore, conventional ground-truth-based segmentation evaluation is not possible. An independently generated image-processing reference mask is used only for an auxiliary pseudo-evaluation. The resulting IoU and Dice values are therefore reported as pseudo-IoU and pseudo-Dice rather than true segmentation accuracy.**

> [!NOTE]
> **The separate 130-image SUT-Crack segmentation dataset and its masks were not used.**

---

## 1. Overview & Research Objective
This experiment evaluates the spatial localization performance of the trained `DualBranchCrackNet` model (`crackdualnet_best.pth`) across four benchmark datasets:
1. **Asphalt 720 (Held-out 30%)**: 1,020 images
2. **Asphalt 360 (Full)**: 8,932 images
3. **SUT-Crack Classification (Full)**: 25,563 images
4. **SDNET2018 (Full)**: 24,334 images

Because these datasets contain image-level classification labels (Cracked / Non-cracked) without pixel-level ground-truth masks, this evaluation pipeline compares:
1. An **independently generated reference mask** ("Pseudo-Ground-Truth / Reference Mask") derived from classical image processing applied directly to the original input image.
2. A **predicted crack localization mask** generated using Grad-CAM backpropagation on the fused feature map (`fused = f_cnn + f_swin` of shape `[B, 512, 16, 16]`) followed by Otsu thresholding.

---

## 2. Methodology

### A. Pseudo-Ground-Truth / Reference Mask Generation
The reference mask is generated independently from the input image without accessing model predictions:
- Grayscale conversion & CLAHE contrast enhancement (`clipLimit=3.0`, `tileGridSize=(8,8)`)
- Bilateral filtering (`d=7`, `sigmaColor=50`, `sigmaSpace=50`) for pavement noise smoothing while preserving crack edges
- Morphological Black Top-Hat filtering (`15x15` rectangular kernel) to isolate dark elongated structures
- Otsu thresholding & Morphological opening (`3x3` ellipse kernel)
- Connected-component filtering removing isolated components $< 20$ pixels

### B. Grad-CAM Predicted Localization Mask
- Model backpropagation targeting class 0 (Crack) logit score
- Gradient capturing on fused feature representation tensor `fused` (`512` channels, `16x16` spatial grid)
- Channel-wise global average pooling of gradients: $\\alpha_k = \\frac{{1}}{{16 \\times 16}} \\sum_{{i,j}} \\frac{{\\partial y^c}}{{\\partial A_{{k,i,j}}}}$
- Weighted sum followed by ReLU, cubic resizing to `256x256`, and Otsu thresholding
- Morphological cleanup (`3x3` ellipse opening and closing)

### C. Mathematical Definitions & Empty-Mask Rules
- **Pseudo-IoU**: $\\frac{{|P \\cap R|}}{{|P \\cup R|}}$
- **Pseudo-Dice**: $\\frac{{2 |P \\cap R|}}{{|P| + |R|}}$
- **SSIM**: Structural Similarity Index calculated between the binary reference and predicted masks ($[0, 255]$).

**Empty Mask Handling Rules**:
- If both reference ($R$) and predicted ($P$) masks contain 0 crack pixels: $\\text{{Pseudo-IoU}} = 1.0$, $\\text{{Pseudo-Dice}} = 1.0$, $\\text{{SSIM}} = 1.0$.
- If only one mask contains 0 crack pixels: $\\text{{Pseudo-IoU}} = 0.0$, $\\text{{Pseudo-Dice}} = 0.0$, and SSIM is calculated normally.

---

## 3. Localization Metrics Summary Table

### Dataset-Wise Localization Metrics
{metrics_markdown}

---

## 4. Rule-Based Crack Severity Distribution

Severity is assigned based on static thresholds applied to the predicted Grad-CAM mask features:
- **High Severity**: Crack Density $\\ge 0.03$ OR Skeleton Length $\\ge 150\\text{{ px}}$
- **Medium Severity**: Crack Density $\\ge 0.01$ OR Skeleton Length $\\ge 50\\text{{ px}}$
- **Low Severity**: Otherwise

{severity_markdown}

---

## 5. Verification & Integrity Checklist
- [x] Checkpoint `training/crackdualnet_best.pth` MD5 verified as `6a005166a7147906bfac621018b378da` (Unmodified).
- [x] Model evaluated strictly in READ-ONLY mode; no retraining or fine-tuning occurred.
- [x] `SUT-Crack/1-Segmentation/` directory was NEVER accessed.
- [x] Reference masks were generated completely independently from model predictions using fixed parameters across all datasets.
- [x] Grad-CAM gradients were captured directly from the fused feature tensor `fused`.
"""
    with open(readme_path, "w") as f:
        f.write(readme_content)

if __name__ == "__main__":
    run_evaluation()


Overwriting experiment_pseudo_segmentation.py


In [6]:
# Step 1: Verify GPU is active
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name    :", torch.cuda.get_device_name(0))

# Step 2: Run the Pseudo-Segmentation Evaluation Script
!python3 experiment_pseudo_segmentation.py


CUDA Available: True
Device Name    : Tesla T4

STARTING PSEUDO-GT & GRAD-CAM EVALUATION PIPELINE

PATH DISCOVERY
  ✓ checkpoint  : /kaggle/input/models/dinjbu/crackdualnet/pytorch/default/1/crackdualnet_best.pth
  ✓ asphalt_720 : /kaggle/input/datasets/dinjbu/asphalt-cracked-and-uncracked-image-dataset-720/Cracked and Uncracked Image Database_720x720
  ✓ asphalt_360 : /kaggle/input/datasets/dinjbu/asphalt-360/Cracked and Uncracked Image Database_360x360
  ✓ sut_crack   : /kaggle/input/datasets/dinjbu/sut-crack/3-Classification
  ✓ sdnet       : /kaggle/input/datasets/dinjbu/sdnet-pavement-2018/SDNET /Pavements
  Checkpoint MD5 : 6a005166a7147906bfac621018b378da
  ✓ Checkpoint MD5 verified successfully.
  Checkpoint MD5 : 6a005166a7147906bfac621018b378da
  ✓ Checkpoint MD5 verified successfully.
  Initialized Parallel Processing Pool with 3 CPU Workers.

EVALUATING DATASET: Asphalt 720 (Held-out 30%)
  Path        : /kaggle/input/datasets/dinjbu/asphalt-cracked-and-uncracked-image-data

In [7]:
import shutil

# 1. Create a zip archive of the evaluation results folder
shutil.make_archive('pseudo_gt_gradcam_results', 'zip', '/kaggle/working/results/pseudo_gt_gradcam_evaluation')

print("✓ Created pseudo_gt_gradcam_results.zip")


✓ Created pseudo_gt_gradcam_results.zip
